# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import trim, col
from pyspark.sql.window import Window

# Reading the Bronze table

In [0]:
df = spark.table("workspace.bronze.erp_px_cat_g1v2")

# Data transformations

In [0]:
df.display()
df.printSchema()

Analyzing the table, nothing really cacthes my eye that can be improved immediately. Maybe we can just rename the columns and trim the table

# Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

# Renaming Columns

In [0]:
RENAME_MAP = {
    "id": "category_id",
    "cat": "category",
    "subcat": "subcategory",
    "maintenance": "maintenance_flag"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

# Sanity check of dataframe

In [0]:
df.limit(10).display()

# Writing Silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_product_category")

# Sanity check of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.erp_product_category LIMIT 10
